# CORD-19 — Schema dei JSON da **parsing XML** (`pmc_json`)

Gemello di `PDF_json_explore.ipynb`, ma per la **seconda sorgente full-text**: i file estratti
dal parsing dell'**XML di PubMed Central** (`document_parses/pmc_json/`, ~112k file `PMCxxxxxx.xml.json`).

Obiettivo: capire lo schema di questo ramo e **confrontarlo con quello PDF**. Anticipiamo la conclusione —
non sono identici: in `pmc_json` **manca `abstract`** e le **affiliation degli autori sono vuote**.
Sono differenze che cambiano quale sorgente usare per ciascun task del progetto.

> Il notebook si aspetta la cartella `archive/` accanto a sé (dentro `MAPD-Project/`).
> Se la sposti, aggiorna `DATA_DIR` più sotto.

## 1. Import

In [1]:
import json
import random
from pathlib import Path
from collections import Counter

import dask
import dask.bag as db
import pandas as pd

print('dask', dask.__version__)

dask 2026.6.0


## Avvio del cluster locale

`LocalCluster` crea scheduler + worker sulla tua macchina (12 core / 24 GB).
Config prudente: 4 worker × 2 thread, 4 GB per worker — gli stessi parametri
su cui poi farai i **benchmark**. Dopo l'avvio, apri la **dashboard** (link sotto).

In [2]:
from dask.distributed import Client, LocalCluster

# chiudi un eventuale cluster precedente prima di crearne uno nuovo
try:
    client.close(); cluster.close()
except NameError:
    pass

cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit='4GB')
client = Client(cluster)
print('Dashboard:', client.dashboard_link)
client

Dashboard: http://127.0.0.1:8787/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 14.90 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:53381,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:53392,Total threads: 2
Dashboard: http://127.0.0.1:53396/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:53384,


2026-07-03 18:05:55,655 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='127.0.0.1:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/Users/federicosbarbati/miniconda/envs/mapd-covid/lib/python3.11/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/federicosbarbati/miniconda/envs/mapd-covid/lib/python3.11/site-packages/tornado/web.py", line 3409, in wrapper
    return method(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/federicosbarbati/miniconda/envs/mapd-covid/lib/python3.11/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token

## 2. Percorsi dei dati

I file `pmc_json` si chiamano `PMC<pmcid>.xml.json`: lo **stem del nome è il `pmcid`** della riga
in `metadata.csv` (mentre nel ramo PDF il nome era lo `sha`). Per esplorare lo schema NON leggiamo
tutti i ~112k file: prendiamo un piccolo campione.

In [3]:
from pprint import pprint

DATA_DIR = Path('archive')
assert DATA_DIR.exists(), f'Non trovo {DATA_DIR.resolve()} — correggi DATA_DIR'

PMC_JSON = DATA_DIR / 'document_parses' / 'pmc_json'
PDF_JSON = DATA_DIR / 'document_parses' / 'pdf_json'   # serve dopo, per il confronto
METADATA = DATA_DIR / 'metadata.csv'

sample_files = sorted(PMC_JSON.glob('PMC*.json'))[:10]
print('file campione:', len(sample_files))
pprint(sample_files)

file campione: 10
[PosixPath('archive/document_parses/pmc_json/PMC1054884.xml.json'),
 PosixPath('archive/document_parses/pmc_json/PMC1065028.xml.json'),
 PosixPath('archive/document_parses/pmc_json/PMC1065064.xml.json'),
 PosixPath('archive/document_parses/pmc_json/PMC1065120.xml.json'),
 PosixPath('archive/document_parses/pmc_json/PMC1065257.xml.json'),
 PosixPath('archive/document_parses/pmc_json/PMC1072802.xml.json'),
 PosixPath('archive/document_parses/pmc_json/PMC1072806.xml.json'),
 PosixPath('archive/document_parses/pmc_json/PMC1072807.xml.json'),
 PosixPath('archive/document_parses/pmc_json/PMC1074505.xml.json'),
 PosixPath('archive/document_parses/pmc_json/PMC1074749.xml.json')]


## 3. Leggere pochi file con una Bag e ispezionare lo schema

⚠️ Come per il PDF, ogni file è **un singolo oggetto JSON pretty-printed** (multi-riga),
non JSON-lines: si legge **un file intero per elemento** con `from_sequence` + `json.load`,
*non* con `db.read_text().map(json.loads)`.

In [4]:
def load_record(path):
    with open(path) as fh:
        return json.load(fh)

files = [str(f) for f in sample_files]
b = db.from_sequence(files).map(load_record)   # bag di dict, uno per paper
record = b.take(1)[0]
print('Top-level keys:', list(record.keys()))

Top-level keys: ['paper_id', 'metadata', 'body_text', 'ref_entries', 'back_matter', 'bib_entries']


### Prima differenza dal PDF: manca `abstract`

Nel ramo PDF le chiavi top-level erano
`['paper_id', 'metadata', 'abstract', 'body_text', 'bib_entries', 'ref_entries', 'back_matter']`.
Qui **`abstract` non c'è**: nei `pmc_json` va sempre letto con `record.get('abstract', [])`,
oppure preso da `metadata.csv`. Il resto dello schema è lo stesso (occhio: `metadata` contiene
**solo** `title` + `authors`; `body_text` & co. sono fratelli top-level, non dentro `metadata`).

In [5]:
print("'abstract' presente? ", 'abstract' in record)

meta = record['metadata']
print('metadata keys :', list(meta.keys()))
print('title         :', meta['title'][:100])
print('n_authors     :', len(meta['authors']))
print('n_body_paragr :', len(record['body_text']))
print()
print('Esempio di un paragrafo di body_text:')
print(json.dumps(record['body_text'][0], indent=2)[:800])

'abstract' presente?  False
metadata keys : ['title', 'authors']
title         : Recombination Every Day: Abundant Recombination in a Virus during a Single Multi-Cellular Host Infec
n_authors     : 7
n_body_paragr : 32

Esempio di un paragrafo di body_text:
{
  "text": "As increasing numbers of full-length viral sequences become available, recombinant or mosaic viruses are being recognized more frequently [1,2,3]. Recombination events have been demonstrated to be associated with viruses expanding their host range [4,5,6,7] or increasing their virulence [8,9], thus accompanying, or perhaps even being at the origin of, major changes during virus adaptation. It remains unclear, however, whether recombination events represent a highly frequent and significant phenomenon in the everyday life of these viruses.",
  "cite_spans": [
    {
      "start": 139,
      "end": 140,
      "mention": "1",
      "ref_id": "BIBREF0"
    },
    {
      "start": 141,
      "end": 142,
      "mention": "2",

## 4. Scansione dello schema su un campione più ampio

Dieci file non bastano a fidarsi. Leggiamo ~300 file a caso e **contiamo la frequenza delle chiavi**:
se lo schema è stabile, ogni chiave top-level deve comparire in tutti i record. Contiamo anche
quante `affiliation` sono effettivamente popolate — è il dato che serve al task *paesi/istituti*.

In [6]:
random.seed(0)
all_pmc = sorted(PMC_JSON.glob('PMC*.json'))
sample = random.sample(all_pmc, 300)
recs = db.from_sequence([str(p) for p in sample]).map(load_record).compute()

top, meta_k, body_k = Counter(), Counter(), Counter()
no_abstract = empty_body = 0
aff_pop = aff_tot = 0
for r in recs:
    top.update(r.keys())
    meta_k.update(r['metadata'].keys())
    if 'abstract' not in r:
        no_abstract += 1
    if not r.get('body_text'):
        empty_body += 1
    for p in r.get('body_text', []):
        body_k.update(p.keys())
    for a in r['metadata'].get('authors', []):
        aff_tot += 1
        if a.get('affiliation'):
            aff_pop += 1

n = len(recs)
print('file letti            :', n)
print('chiavi top-level      :', dict(top))
print('chiavi metadata       :', dict(meta_k))
print('chiavi paragrafo      :', dict(body_k))
print('record senza abstract :', f'{no_abstract}/{n}')
print('record con body vuoto :', f'{empty_body}/{n}')
print('affiliation popolate  :', f'{aff_pop}/{aff_tot}')

file letti            : 300
chiavi top-level      : {'paper_id': 300, 'metadata': 300, 'body_text': 300, 'ref_entries': 300, 'back_matter': 300, 'bib_entries': 300}
chiavi metadata       : {'title': 300, 'authors': 300}
chiavi paragrafo      : {'text': 8465, 'cite_spans': 8465, 'section': 8465, 'ref_spans': 8465}
record senza abstract : 300/300
record con body vuoto : 0/300
affiliation popolate  : 0/1918


### Seconda differenza: le `affiliation` sono vuote in `pmc_json`

Nel campione le affiliation popolate sono **0 su tutte**: il parser XML di PMC riempie
`authors[i].affiliation` con `{}`. Nel ramo PDF, invece, ~metà degli autori ha un'affiliation.

**Conseguenza pratica per il Task 2 (paesi/istituti più/meno rappresentati): usa `pdf_json`,
non `pmc_json`.** Lo verifichiamo esplicitamente nel confronto qui sotto.

## 5. Confronto diretto PDF vs PMC

Profiliamo un campione di ciascuna sorgente e li mettiamo fianco a fianco. È il cuore di questo
notebook: rende esplicite le differenze di schema e di *qualità* tra i due rami full-text.

In [8]:
def profile(paths):
    recs = [load_record(str(p)) for p in paths]
    n = len(recs)
    aff_tot = sum(len(r['metadata'].get('authors', [])) for r in recs)
    aff_pop = sum(1 for r in recs for a in r['metadata'].get('authors', [])
                  if a.get('affiliation'))
    keys = set().union(*[set(r.keys()) for r in recs])
    return {
        'file profilati'     : n,
        'ha abstract'        : f"{sum('abstract' in r for r in recs)}/{n}",
        'body_text vuoto'    : f"{sum(not r.get('body_text') for r in recs)}/{n}",
        'authors vuoti'      : f"{sum(not r['metadata'].get('authors') for r in recs)}/{n}",
        'affiliation popolate': f'{aff_pop}/{aff_tot}',
        'top-level keys'     : sorted(keys),
    }

random.seed(1)
pdf_sample = random.sample(sorted(PDF_JSON.glob('*.json')), 150)
pmc_sample = random.sample(all_pmc, 150)

cmp = pd.DataFrame({'pdf_json': profile(pdf_sample), 'pmc_json': profile(pmc_sample)})
cmp

,pdf_json,pmc_json
file profilati,150,150
ha abstract,150/150,0/150
body_text vuoto,0/150,0/150
authors vuoti,11/150,0/150
affiliation popolate,507/973,0/998
top-level keys,"[abstract, back_matter, bib_entries, body_text...","[back_matter, bib_entries, body_text, metadata..."


In [9]:
# differenza di chiavi, resa esplicita
kp = set(cmp.loc['top-level keys', 'pdf_json'])
km = set(cmp.loc['top-level keys', 'pmc_json'])
print('solo nel PDF :', kp - km or '—')
print('solo nel PMC :', km - kp or '—')
print('in comune    :', sorted(kp & km))

solo nel PDF : {'abstract'}
solo nel PMC : —
in comune    : ['back_matter', 'bib_entries', 'body_text', 'metadata', 'paper_id', 'ref_entries']


## 6. Full-text di un paper

Come per il PDF, il testo completo per il word-count (Task 1) si ottiene concatenando i `text`
dei paragrafi in `body_text`. Nota: nel campione `pmc_json` il `body_text` è **sempre presente**
(0 vuoti), tipicamente più pulito/completo del parse da PDF.

In [10]:
full_text = ' '.join(p['text'] for p in record['body_text'])
print(len(full_text), 'caratteri')
print(full_text[:500], '...')

30048 caratteri
As increasing numbers of full-length viral sequences become available, recombinant or mosaic viruses are being recognized more frequently [1,2,3]. Recombination events have been demonstrated to be associated with viruses expanding their host range [4,5,6,7] or increasing their virulence [8,9], thus accompanying, or perhaps even being at the origin of, major changes during virus adaptation. It remains unclear, however, whether recombination events represent a highly frequent and significant pheno ...


## 7. Cosa significa per l'analisi (conclusioni)

Confronto sintetico tra i due rami full-text, con le ricadute sui task del progetto:

| aspetto | `pdf_json` | `pmc_json` (XML) |
|---|---|---|
| chiave file | `sha` (nome = `<sha>.json`) | `pmcid` (nome = `PMC<pmcid>.xml.json`) |
| `abstract` | **presente** | **assente** → `get('abstract', [])` o da `metadata.csv` |
| `body_text` | a volte vuoto | **quasi sempre presente**, più pulito |
| `affiliation` autori | ~50% popolate | **~0% popolate** |
| conteggio | ~151k file | ~112k file |

- **Task 1 — word count sul full-text:** `pmc_json` è un'ottima base (body sempre presente). Per
  massima copertura si uniscono i due rami, ma i paper con **entrambe** le parse (~105k) vanno
  **deduplicati** su `cord_uid` (vedi `metadata_health.ipynb`), preferendo la parse PMC.
- **Task 2 — paesi/istituti:** le affiliation servono e in `pmc_json` sono vuote → **usa `pdf_json`**
  (`record['metadata']['authors'][i]['affiliation']`).
- **Task 3/4 — embeddings/similarità dei titoli:** i titoli stanno già in `metadata.csv`, non serve
  aprire i JSON.
- `bib_entries` (`title, authors, year, venue, volume, issn, pages, other_ids`) e `ref_entries`
  (`text, type`) sono disponibili se in futuro servisse un'analisi di citazioni.

## 8. Chiusura

Chiudi il cluster quando hai finito (libera memoria e porte).

In [11]:
client.close()
cluster.close()